## Step 1 - Load raw tables

In [6]:
import pandas as pd
import numpy as np

from pathlib import Path
data_dir = Path().resolve().parent / 'data' / 'raw'

members  = pd.read_csv(data_dir / 'members.csv')
benefits = pd.read_csv(data_dir / 'benefits.csv')
claims   = pd.read_csv(data_dir / 'claims.csv')

print(f"Members:  {len(members):,}")
print(f"Benefits: {len(benefits):,}")
print(f"Claims:   {len(claims):,}")

assert set(benefits['member_id'].unique()) == set(members['member_id']), "Benefits member_ids mismatch"
assert set(claims['member_id'].unique()) == set(members['member_id']), "Claims member_ids mismatch"

Members:  50,000
Benefits: 200,000
Claims:   50,000


In [7]:
print(members[condition_cols].isnull().sum())
print(f"Total members: {len(members)}")


has_msk_flag          0
has_metabolic_flag    0
has_mh_flag           0
comorbidity_count     0
condition_cluster     0
dtype: int64
Total members: 50000


## Step 2 - Demographic Features

In [8]:
features = members[['member_id', 'age', 'gender', 'plan_type',
                     'employer_group_size', 'tenure_months']].copy()

features['age_band'] = pd.cut(
    features['age'],
    bins=[24, 34, 44, 54, 65],
    labels=['25-34', '35-44', '45-54', '55-65']
).astype(str)

print("Age band distribution:")
print(features['age_band'].value_counts().sort_index())

Age band distribution:
age_band
25-34    11406
35-44    18605
45-54    14701
55-65     5288
Name: count, dtype: int64


## Step -3 Condition Flags

In [9]:
condition_cols = ['has_msk_flag', 'has_metabolic_flag', 'has_mh_flag',
                  'comorbidity_count', 'condition_cluster']

features = features.merge(members[['member_id'] + condition_cols], on='member_id', how='left')

# Find which members are missing condition data
missing = features[features[condition_cols].isnull().any(axis=1)]['member_id']
print(f"{len(missing)} members missing condition data")
print("Sample missing IDs:", missing.head().tolist())

# Check if they exist in members at all
in_members = missing.isin(members['member_id'])
print(f"  {in_members.sum()} found in members (null condition values)")
print(f"  {(~in_members).sum()} not found in members at all")


assert features[condition_cols].isnull().sum().sum() == 0, "Null condition flags"
print("Condition cluster distribution:")
print(features['condition_cluster'].value_counts())

0 members missing condition data
Sample missing IDs: []
  0 found in members (null condition values)
  0 not found in members at all
Condition cluster distribution:
condition_cluster
Metabolic    21371
Mixed        14056
Healthy       9565
MSK           3814
MH            1194
Name: count, dtype: int64


## Step 4 - Claims History Features

In [ ]:
features = features.merge(claims, on='member_id', how='left')

claims_cols = ['gp_visits_6m', 'specialist_visits_6m', 'ed_visits_6m',
               'allied_health_claims_6m', 'days_since_last_allied',
               'total_claims_6m', 'total_spend_6m']
features[claims_cols] = features[claims_cols].fillna(0)
features['days_since_last_allied'] = features['days_since_last_allied'].replace(0, 999)

# GP-to-specialist ratio (0 if no GP visits)
features['gp_to_specialist_ratio'] = np.where(
    features['gp_visits_6m'] > 0,
    features['specialist_visits_6m'] / features['gp_visits_6m'],
    0.0
).round(3)

# Zero allied health flag
features['zero_allied_health_flag'] = (features['allied_health_claims_6m'] == 0).astype(int)

# High GP + low allied composite signal
features['high_gp_low_allied'] = (
    (features['gp_visits_6m'] > 4) & (features['allied_health_claims_6m'] == 0)
).astype(int)

print(f"Zero allied health (6m): {features['zero_allied_health_flag'].mean():.1%}")
print(f"High GP + low allied:    {features['high_gp_low_allied'].mean():.1%}")

## Step 5 - Benefits Utilisation Features

In [11]:
benefits_wide = benefits.pivot_table(
    index='member_id',
    columns='benefit_type',
    values=['sessions_entitled', 'sessions_used', 'sessions_remaining'],
    aggfunc='first'
)
benefits_wide.columns = ['_'.join(col).strip() for col in benefits_wide.columns]
benefits_wide = benefits_wide.reset_index()

features = features.merge(benefits_wide, on='member_id', how='left')

total_entitled_cols = [c for c in benefits_wide.columns if c.startswith('sessions_entitled_')]
total_used_cols     = [c for c in benefits_wide.columns if c.startswith('sessions_used_')]
remaining_cols      = [c for c in benefits_wide.columns if c.startswith('sessions_remaining_')]

features['total_sessions_entitled'] = features[total_entitled_cols].sum(axis=1)
features['total_sessions_used']     = features[total_used_cols].sum(axis=1)
features['sessions_remaining_total'] = features[remaining_cols].sum(axis=1)

features['benefit_utilisation_rate'] = (
    features['total_sessions_used'] / features['total_sessions_entitled']
).round(4).clip(0, 1)

features['allied_health_utilisation_rate'] = features['benefit_utilisation_rate']
features['any_benefits_remaining'] = (features['sessions_remaining_total'] > 0).astype(int)

print(f"Members with benefits remaining: {features['any_benefits_remaining'].mean():.1%}")
print(f"Avg benefit utilisation rate:    {features['benefit_utilisation_rate'].mean():.2f}")

Members with benefits remaining: 100.0%
Avg benefit utilisation rate:    0.26


## Step 6 - Final column selection and encoding

In [ ]:
FEATURE_COLS = [
    'age', 'age_band', 'gender', 'plan_type', 'employer_group_size', 'tenure_months',
    'has_msk_flag', 'has_metabolic_flag', 'has_mh_flag',
    'comorbidity_count', 'condition_cluster',
    'total_claims_6m', 'total_spend_6m',
    'gp_visits_6m', 'specialist_visits_6m', 'allied_health_claims_6m',
    'days_since_last_allied', 'allied_health_utilisation_rate',
    'gp_to_specialist_ratio', 'zero_allied_health_flag', 'high_gp_low_allied',
    'sessions_remaining_physio', 'sessions_remaining_chiro',
    'sessions_remaining_dietetics', 'sessions_remaining_psychology',
    'any_benefits_remaining', 'benefit_utilisation_rate',
]

output_df = features[['member_id'] + FEATURE_COLS].copy()

CAT_COLS = ['age_band', 'gender', 'plan_type', 'employer_group_size', 'condition_cluster']
for col in CAT_COLS:
    output_df[col] = output_df[col].astype('category')

print(f"Final feature matrix shape: {output_df.shape}")
nulls = output_df.isnull().sum()
print(f"Columns with nulls: {nulls[nulls > 0].to_dict()}")

## Step 7 - Save parquet and load to v530s postgres

In [15]:
import os
from sqlalchemy import create_engine

features_dir = Path().resolve().parent / 'data' / 'features'
os.makedirs(features_dir, exist_ok=True)
output_df.to_parquet(features_dir / 'features.parquet', index=False)
print(f"Saved: {features_dir / 'features.parquet'}")

# Tailscale (preferred) — stable across all networks
V530S_IP = '100.68.3.15'
# Local LAN fallback — uncomment if Tailscale unavailable
# V530S_IP = '192.168.x.x'
engine = create_engine(f"postgresql://ds_user:password@{V530S_IP}:5432/allied_health")
output_df.to_sql('features', engine, schema='allied_health', if_exists='replace', index=False)
print(f"Loaded {len(output_df):,} rows to allied_health.features on V530s")

Saved: C:\Users\alexl\OneDrive\Documents\Personal Projects\allied-health-nudge\data\features\features.parquet
Loaded 50,000 rows to allied_health.features on V530s


## Validation Checks

In [16]:
features_check = pd.read_parquet(Path().resolve().parent / 'data' / 'features' / 'features.parquet')

assert features_check.shape == (50_000, len(FEATURE_COLS) + 1), f"Shape mismatch: {features_check.shape}"
assert features_check['member_id'].nunique() == 50_000, "Duplicate member_ids"

null_pct = features_check.isnull().mean()
assert (null_pct > 0.20).sum() == 0, f"Features with >20% nulls: {null_pct[null_pct>0.20].to_dict()}"

assert features_check['allied_health_utilisation_rate'].between(0, 1).all(), "Utilisation rate out of range"
assert features_check['benefit_utilisation_rate'].between(0, 1).all(), "Benefit util rate out of range"
assert features_check['comorbidity_count'].between(0, 3).all(), "Comorbidity count out of range"
assert features_check['gp_to_specialist_ratio'].ge(0).all(), "Negative GP ratio"

# days_since_last_allied should be 1–180 or exactly 999
valid_days = features_check['days_since_last_allied'].between(1, 180) | \
             (features_check['days_since_last_allied'] == 999)
assert valid_days.all(), "Unexpected days_since_last_allied values"

print("✅ All Stage 2 validation checks passed")
print(f"Feature matrix: {features_check.shape[0]:,} rows × {features_check.shape[1]} columns")

✅ All Stage 2 validation checks passed
Feature matrix: 50,000 rows × 28 columns
